# 18 | LangGraph Streaming：用 v2 格式看见图每一步怎么跑

前面的实验大多使用 `invoke()`：输入初始 State，等整张图跑完，再一次性拿到最终 State。

这对简单流程够用，但真实 Agent 往往不是一口气给出答案。它会先收集信息，再检查风险，再调用模型分析，最后生成结果。只看最终输出，很难知道中间哪一步出了问题。

这一篇用一个安全巡检流水线来观察 streaming：

```text
输入资产
 -> 加载资产画像
 -> 检查暴露端口
 -> 检查漏洞线索
 -> 检查弱口令风险
 -> 调用本地 LLM 汇总风险
 -> 生成巡检报告
```

这条流水线的过程本身就很重要：我们希望边运行边看到当前检查到哪一步、State 写入了什么、模型是否正在输出。

## 一、这篇先解决什么问题

这篇只讲 LangGraph streaming 的基础观察方法，不讨论并行扫描、真实漏洞库、真实端口扫描和前端 SSE。

重点是看懂四种常见 stream mode：

| mode | 看见什么 | 适合什么时候用 |
| --- | --- | --- |
| `updates` | 每个节点返回的状态增量 | 调试节点到底改了什么 |
| `values` | 每一步之后的完整 State | 检查状态整体是否符合预期 |
| `custom` | 节点主动推送的自定义事件 | 展示进度、阶段、审计信息 |
| `messages` | 模型流式输出的消息片段 | 观察 LLM token 输出 |

本系列默认使用 `version="v2"`，因为它把不同 mode 的输出统一成同一种事件结构。

## 二、准备依赖和本地模型

本实验会调用本地 Ollama 中的 `qwen3-coder:30b`。

运行前先确认 Ollama 服务已启动，并且本机已经有这个模型：

```bash
ollama list
```

如果没有，可以先拉取：

```bash
ollama pull qwen3-coder:30b
```

这里的安全检查数据是实验用的固定样例，不会访问真实网络。LLM 只负责把前面节点收集到的发现整理成风险判断。

In [15]:
from __future__ import annotations

from importlib.metadata import version
from pprint import pprint
from typing import TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_ollama import ChatOllama
from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph

print("langgraph", version("langgraph"))
print("langchain-ollama", version("langchain-ollama"))

langgraph 1.2.0
langchain-ollama 1.1.0


模型参数刻意设得保守一点：`temperature=0` 让输出稳定，`num_predict=160` 控制回答长度，避免基础实验里生成太久。

In [16]:
MODEL_NAME = "qwen3-coder:30b"

# 这里使用本地 Ollama 模型。temperature=0 让同一输入下输出尽量稳定，
# num_predict=160 用来限制输出长度，避免基础实验里等太久。
llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0,
    num_predict=160,
)

## 三、定义安全巡检 State

这张图的 State 不只保存最终回答，而是保存巡检过程中逐步形成的业务字段：

- `asset_id`：要检查的资产；
- `asset_profile`：资产画像；
- `exposure_findings`：暴露面检查结果；
- `vulnerability_findings`：漏洞线索；
- `credential_findings`：弱口令或凭据风险；
- `risk_summary`：LLM 汇总出的风险判断；
- `final_report`：最终巡检报告。

后面看 `updates` 和 `values` 时，这些字段会一点点被写入。

In [17]:
# State 描述这张图会维护哪些业务字段。
# 后面的节点不会互相直接传参，而是统一读写这份 State。
class SecurityScanState(TypedDict):
    # 输入资产 ID，图启动时就有。
    asset_id: str
    # 资产画像，由 load_asset_profile 节点写入。
    asset_profile: str
    # 暴露面检查结果，由 check_exposure 节点写入。
    exposure_findings: list[str]
    # 漏洞线索，由 check_vulnerabilities 节点写入。
    vulnerability_findings: list[str]
    # 凭据和登录面风险，由 check_credentials 节点写入。
    credential_findings: list[str]
    # LLM 基于前面发现生成的风险判断。
    risk_summary: str
    # 最终报告，由 build_final_report 节点写入。
    final_report: str

## 四、定义巡检节点

前四个节点使用固定样例数据，模拟资产中心、暴露面系统、漏洞情报和凭据检查结果。

每个节点都会调用 `get_stream_writer()` 推送进度事件。这些事件不会写入 State，但可以被 `custom` mode 看见。

In [18]:
def load_asset_profile(state: SecurityScanState) -> dict[str, str]:
    # 节点 1：模拟从资产中心读取资产画像。
    # writer({...}) 只发 custom 过程事件，不写入 State。
    writer = get_stream_writer()
    writer({"stage": "asset", "message": f"读取资产 {state['asset_id']} 的基础画像"})

    # return {...} 才是写回 State 的内容。
    # 开启 stream_mode="updates" 后，可以看到 asset_profile 被这个节点写入。
    profile = "公网 Web 服务器，业务系统：供应商门户，环境：生产，责任团队：应用安全组"
    return {"asset_profile": profile}


def check_exposure(state: SecurityScanState) -> dict[str, list[str]]:
    # 节点 2：模拟暴露面检查，例如公网端口、管理入口、调试端口。
    writer = get_stream_writer()
    writer({"stage": "exposure", "message": "检查公网暴露端口"})

    # 这些 findings 是后续风险汇总要读取的业务数据，所以要写入 State。
    findings = [
        "443/tcp 对公网开放，运行 HTTPS 服务",
        "22/tcp 对办公出口开放，存在远程管理入口",
        "8080/tcp 对公网开放，疑似调试控制台",
    ]
    return {"exposure_findings": findings}


def check_vulnerabilities(state: SecurityScanState) -> dict[str, list[str]]:
    # 节点 3：模拟漏洞情报和组件版本检查。
    # 这里不做真实扫描，只用固定样例保持实验可复现。
    writer = get_stream_writer()
    writer({"stage": "vulnerability", "message": "匹配漏洞情报和组件版本"})

    findings = [
        "检测到 Nginx 版本落后于安全基线 2 个小版本",
        "供应商门户登录页缺少基础速率限制证据",
        "8080 控制台返回默认标题，可能暴露管理界面",
    ]
    return {"vulnerability_findings": findings}


def check_credentials(state: SecurityScanState) -> dict[str, list[str]]:
    # 节点 4：模拟凭据和登录面风险检查。
    # 这些信息会和前面的暴露面、漏洞线索一起交给 LLM 汇总。
    writer = get_stream_writer()
    writer({"stage": "credential", "message": "检查凭据和登录面风险"})

    findings = [
        "发现 admin 用户名存在性提示",
        "登录失败响应区分账号不存在和密码错误",
    ]
    return {"credential_findings": findings}

第五个节点调用本地 LLM。它读取前面节点写入的检查结果，让模型输出一段简短风险判断。

注意：这个节点不是让模型重新扫描资产，而是让模型基于已知发现做汇总。图结构负责控制检查流程，模型负责语言化分析。

In [19]:
def summarize_risk_with_llm(state: SecurityScanState) -> dict[str, str]:
    # 节点 5：调用本地 LLM 做风险汇总。
    # 注意：模型不负责扫描资产，只负责基于前面节点写入 State 的事实做总结。
    writer = get_stream_writer()
    writer({"stage": "llm", "message": f"调用本地 {MODEL_NAME} 汇总风险"})

    # prompt 只使用 State 中已有字段，避免让模型凭空编造新发现。
    prompt = f"""
请基于以下安全巡检结果，输出一段简短风险判断。

要求：
1. 只基于给定事实，不编造额外发现。
2. 用中文回答。
3. 控制在 3 条要点内。
4. 每条包含：风险、原因、建议动作。

资产：{state['asset_id']}
资产画像：{state['asset_profile']}

暴露面发现：
{chr(10).join('- ' + item for item in state['exposure_findings'])}

漏洞线索：
{chr(10).join('- ' + item for item in state['vulnerability_findings'])}

凭据风险：
{chr(10).join('- ' + item for item in state['credential_findings'])}
""".strip()

    # 这里使用 invoke 得到完整回复；外层如果开启 stream_mode="messages"，
    # LangGraph 仍然可以捕获模型的 token 流式事件。
    response = llm.invoke(
        [
            SystemMessage(content="你是企业安全巡检助手，回答必须克制、具体、可执行。"),
            HumanMessage(content=prompt),
        ]
    )
    return {"risk_summary": response.content}


def build_final_report(state: SecurityScanState) -> dict[str, str]:
    # 节点 6：把前面所有发现和 LLM 风险判断拼成最终报告。
    # 这个节点不再做新判断，只负责组织输出格式。
    writer = get_stream_writer()
    writer({"stage": "report", "message": "生成最终巡检报告"})

    report = f"""
资产：{state['asset_id']}
画像：{state['asset_profile']}

暴露面发现：
{chr(10).join('- ' + item for item in state['exposure_findings'])}

漏洞线索：
{chr(10).join('- ' + item for item in state['vulnerability_findings'])}

凭据风险：
{chr(10).join('- ' + item for item in state['credential_findings'])}

LLM 风险判断：
{state['risk_summary']}
""".strip()
    return {"final_report": report}

## 五、组装 Graph

这篇先用线性流程，故意不引入条件边和并行。这样可以把注意力集中在 streaming 事件结构上。

后面学习 `Send` 时，再把暴露面、漏洞、凭据检查改成并行分发。

In [20]:
# 用 SecurityScanState 作为这张图的状态结构。
builder = StateGraph(SecurityScanState)

# 注册节点：节点名会出现在 updates 事件里。
builder.add_node("load_asset_profile", load_asset_profile)
builder.add_node("check_exposure", check_exposure)
builder.add_node("check_vulnerabilities", check_vulnerabilities)
builder.add_node("check_credentials", check_credentials)
builder.add_node("summarize_risk_with_llm", summarize_risk_with_llm)
builder.add_node("build_final_report", build_final_report)

# 这篇先用线性流程：每个节点执行完，固定进入下一个节点。
# 后续学习条件边和 Send 时，再引入分支、循环和并行。
builder.add_edge(START, "load_asset_profile")
builder.add_edge("load_asset_profile", "check_exposure")
builder.add_edge("check_exposure", "check_vulnerabilities")
builder.add_edge("check_vulnerabilities", "check_credentials")
builder.add_edge("check_credentials", "summarize_risk_with_llm")
builder.add_edge("summarize_risk_with_llm", "build_final_report")
builder.add_edge("build_final_report", END)

# compile 之后才得到真正可运行的 graph。
graph = builder.compile()

In [21]:
# 初始 State：输入资产 ID 已知，其余字段先放空。
# 运行过程中，每个节点会把自己的检查结果逐步写回这些字段。
input_state = {
    "asset_id": "asset-prod-supplier-portal-01",
    "asset_profile": "",
    "exposure_findings": [],
    "vulnerability_findings": [],
    "credential_findings": [],
    "risk_summary": "",
    "final_report": "",
}

## 六、invoke：只能看到最终结果

`invoke()` 会等整张图跑完，再返回最终 State。

它适合拿结果，但不适合观察过程。这里第一次运行会调用本地 LLM，如果模型还没加载，可能需要稍等。

In [22]:
# invoke 会一次性跑完整张图，只返回最终 State。
# 它适合拿最终结果，但看不见节点执行过程。
final_state = graph.invoke(input_state)
print(final_state["final_report"])

资产：asset-prod-supplier-portal-01
画像：公网 Web 服务器，业务系统：供应商门户，环境：生产，责任团队：应用安全组

暴露面发现：
- 443/tcp 对公网开放，运行 HTTPS 服务
- 22/tcp 对办公出口开放，存在远程管理入口
- 8080/tcp 对公网开放，疑似调试控制台

漏洞线索：
- 检测到 Nginx 版本落后于安全基线 2 个小版本
- 供应商门户登录页缺少基础速率限制证据
- 8080 控制台返回默认标题，可能暴露管理界面

凭据风险：
- 发现 admin 用户名存在性提示
- 登录失败响应区分账号不存在和密码错误

LLM 风险判断：
**风险要点1：**
- **风险**：Nginx版本存在已知安全漏洞风险
- **原因**：检测到Nginx版本落后于安全基线2个小版本
- **建议动作**：立即升级Nginx至符合安全基线的版本

**风险要点2：**
- **风险**：管理界面存在未授权访问风险
- **原因**：8080端口开放且返回默认标题，暴露调试控制台
- **建议动作**：关闭8080端口对外访问权限或加强访问控制

**风险要点3：**
- **风险**：登录认证存在账户枚举风险
- **原因**：发现admin用户名存在性提示，登录失败响应未做区分


最终报告能说明图跑完了，但看不出中间哪个节点先写入了什么。下面改用 streaming。

## 七、updates：看每个节点写回了什么

`updates` 是最常用的调试模式。它不会打印完整 State，只返回每个节点本次写回的状态增量。

这能直接回答一个问题：**刚刚这个节点到底改了什么？**

In [23]:
# updates 只看节点 return {...} 写回 State 的增量。
# 每条 event 的 data 形如：{"节点名": {"字段": "新值"}}。
for event in graph.stream(input_state, stream_mode="updates", version="v2"): # type: ignore
    print("type:", event["type"], "ns:", event["ns"])
    pprint(event["data"])
    print("-" * 80)

type: updates ns: ()
{'load_asset_profile': {'asset_profile': '公网 Web '
                                         '服务器，业务系统：供应商门户，环境：生产，责任团队：应用安全组'}}
--------------------------------------------------------------------------------
type: updates ns: ()
{'check_exposure': {'exposure_findings': ['443/tcp 对公网开放，运行 HTTPS 服务',
                                          '22/tcp 对办公出口开放，存在远程管理入口',
                                          '8080/tcp 对公网开放，疑似调试控制台']}}
--------------------------------------------------------------------------------
type: updates ns: ()
{'check_vulnerabilities': {'vulnerability_findings': ['检测到 Nginx 版本落后于安全基线 2 '
                                                      '个小版本',
                                                      '供应商门户登录页缺少基础速率限制证据',
                                                      '8080 '
                                                      '控制台返回默认标题，可能暴露管理界面']}}
--------------------------------------------------------------------------------

`version="v2"` 后，每条事件都有统一结构：

```python
{
    "type": "updates",
    "ns": (),
    "data": {"node_name": {"field": "value"}},
}
```

- `type` 表示事件类型，也就是当前 stream mode；
- `ns` 表示 namespace，后面学习子图时会派上用场；
- `data` 才是真正的数据。

## 八、custom：看节点主动推送的进度

安全巡检很适合演示 `custom`：有些信息只是运行过程中的进度，不一定要进入最终 State。

例如“正在检查公网暴露端口”“正在调用本地模型汇总风险”，这些更像进度事件。

In [24]:
# custom 只看节点里 writer({...}) 主动发出的过程事件。
# 这里不会展示 return 写回的 State 更新。
for event in graph.stream(input_state, stream_mode="custom", version="v2"): # type: ignore
    print(f"[{event['data']['stage']}] {event['data']['message']}")

[asset] 读取资产 asset-prod-supplier-portal-01 的基础画像
[exposure] 检查公网暴露端口
[vulnerability] 匹配漏洞情报和组件版本
[credential] 检查凭据和登录面风险
[llm] 调用本地 qwen3-coder:30b 汇总风险
[report] 生成最终巡检报告


`custom` 事件不会自动改变 State。它更像运行过程中的旁路通知。

注意：`get_stream_writer()` 来自 `langgraph.config`，不是后面要讲的 `runtime.context`。这两个概念不要混在一起。

## 九、messages：看本地 LLM 逐 token 输出

`messages` 用来观察聊天模型的流式输出。

这张图只有 `summarize_risk_with_llm` 节点会调用模型，所以 `messages` 事件应该主要来自这个节点。

In [29]:
# messages 用来观察聊天模型的 token 流式输出。
# event["data"] 通常是 (message_chunk, metadata)。
for event in graph.stream(input_state, stream_mode="messages", version="v2"): # type: ignore
    chunk, metadata = event["data"]
    if chunk.content:
        print(chunk.content, end="", flush=True)

print()

**风险要点1：**
- **风险**：Nginx版本存在已知安全漏洞风险
- **原因**：检测到Nginx版本落后于安全基线2个小版本
- **建议动作**：立即升级Nginx至符合安全基线的版本

**风险要点2：**
- **风险**：管理界面存在未授权访问风险
- **原因**：8080端口开放且返回默认标题，暴露调试控制台
- **建议动作**：关闭8080端口对外访问权限或加强访问控制

**风险要点3：**
- **风险**：登录认证机制存在弱口令攻击风险
- **原因**：发现admin用户名提示且登录失败响应未做区分


`messages` 事件的 `data` 通常是 `(message_chunk, metadata)`：

- `message_chunk` 是模型流式输出的一小段内容；
- `metadata` 里能看到事件来自哪个 LangGraph 节点。

如果想确认 token 来自哪个节点，可以打印 metadata。

In [ ]:
token_count = 0

# 这里打印前 8 个 token 的 metadata，确认它们来自哪个 LangGraph 节点。
for event in graph.stream(input_state, stream_mode="messages", version="v2"): # type: ignore
    chunk, metadata = event["data"]
    if not chunk.content:
        continue

    token_count += 1
    if token_count <= 8:
        print(
            {
                "token": chunk.content,
                "node": metadata.get("langgraph_node"),
                "step": metadata.get("langgraph_step"),
            }
        )

{'token': '**', 'node': 'summarize_risk_with_llm', 'step': 5}
{'token': '风险', 'node': 'summarize_risk_with_llm', 'step': 5}
{'token': '要点', 'node': 'summarize_risk_with_llm', 'step': 5}
{'token': '1', 'node': 'summarize_risk_with_llm', 'step': 5}
{'token': '：', 'node': 'summarize_risk_with_llm', 'step': 5}
{'token': '**', 'node': 'summarize_risk_with_llm', 'step': 5}
{'token': '\n-', 'node': 'summarize_risk_with_llm', 'step': 5}
{'token': ' **', 'node': 'summarize_risk_with_llm', 'step': 5}


## 十、values：看每一步之后的完整 State

`values` 会在每一步之后返回完整 State。它比 `updates` 更啰嗦，但适合检查状态整体是否符合预期。

这里为了避免输出太长，只打印每一步里几个字段的长度和是否已经生成风险总结。

In [30]:
# values 会在每一步后返回完整 State。
# 完整报告很长，所以这里只抽取几个字段做快照，观察 State 如何逐步成形。
for event in graph.stream(input_state, stream_mode="values", version="v2"): # type: ignore
    state = event["data"]
    snapshot = {
        "asset_profile_ready": bool(state["asset_profile"]),
        "exposure_count": len(state["exposure_findings"]),
        "vulnerability_count": len(state["vulnerability_findings"]),
        "credential_count": len(state["credential_findings"]),
        "risk_summary_ready": bool(state["risk_summary"]),
        "final_report_ready": bool(state["final_report"]),
    }
    pprint(snapshot)

{'asset_profile_ready': False,
 'credential_count': 0,
 'exposure_count': 0,
 'final_report_ready': False,
 'risk_summary_ready': False,
 'vulnerability_count': 0}
{'asset_profile_ready': True,
 'credential_count': 0,
 'exposure_count': 0,
 'final_report_ready': False,
 'risk_summary_ready': False,
 'vulnerability_count': 0}
{'asset_profile_ready': True,
 'credential_count': 0,
 'exposure_count': 3,
 'final_report_ready': False,
 'risk_summary_ready': False,
 'vulnerability_count': 0}
{'asset_profile_ready': True,
 'credential_count': 0,
 'exposure_count': 3,
 'final_report_ready': False,
 'risk_summary_ready': False,
 'vulnerability_count': 3}
{'asset_profile_ready': True,
 'credential_count': 2,
 'exposure_count': 3,
 'final_report_ready': False,
 'risk_summary_ready': False,
 'vulnerability_count': 3}
{'asset_profile_ready': True,
 'credential_count': 2,
 'exposure_count': 3,
 'final_report_ready': False,
 'risk_summary_ready': True,
 'vulnerability_count': 3}
{'asset_profile_ready'

一个简单判断：

- 想知道节点返回了什么，用 `updates`；
- 想知道当前完整状态是什么，用 `values`。

## 十一、多个 mode：为什么推荐 v2 格式

真实调试时，我们经常想同时看进度事件、节点状态更新和模型 token。

使用 `version="v2"` 后，不管开一个 mode 还是多个 mode，事件结构都保持一致。解析代码只需要看 `event["type"]`。

In [ ]:
# 多个 mode 可以同时开启。
# version="v2" 后，不管事件来自 custom、updates 还是 messages，
# 都可以统一通过 event["type"] 判断类型，再处理 event["data"]。
for event in graph.stream(
    input_state, # type: ignore
    stream_mode=["custom", "updates", "messages"],
    version="v2",
): # type: ignore
    if event["type"] == "custom":
        print(f"进度：{event['data']['message']}")
    elif event["type"] == "updates":
        node_name = next(iter(event["data"]))
        print(f"\n状态更新：{node_name}")
    elif event["type"] == "messages":
        chunk, _metadata = event["data"]
        if chunk.content:
            print(chunk.content, end="", flush=True)

print()

进度：读取资产 asset-prod-supplier-portal-01 的基础画像

状态更新：load_asset_profile
进度：检查公网暴露端口

状态更新：check_exposure
进度：匹配漏洞情报和组件版本

状态更新：check_vulnerabilities
进度：检查凭据和登录面风险

状态更新：check_credentials
进度：调用本地 qwen3-coder:30b 汇总风险
**风险要点1：**
- **风险**：Nginx版本存在已知安全漏洞风险
- **原因**：检测到Nginx版本落后于安全基线2个小版本
- **建议动作**：立即升级Nginx至符合安全基线的版本

**风险要点2：**
- **风险**：管理界面存在未授权访问风险
- **原因**：8080端口开放且返回默认标题，暴露调试控制台
- **建议动作**：关闭8080端口对外访问权限或加强访问控制

**风险要点3：**
- **风险**：登录认证机制存在弱口令攻击风险
- **原因**：发现admin用户名提示且登录失败响应未做区分
状态更新：summarize_risk_with_llm
进度：生成最终巡检报告

状态更新：build_final_report



这就是 v2 格式的核心价值：**调用方式可以变复杂，事件解析逻辑不需要跟着变形。**

后面学习子图、多 Agent 和前端 streaming 时，这个习惯会很省心。

## 十二、小结

这一篇只需要记住四个判断：

| 你想看什么 | 用什么 mode |
| --- | --- |
| 节点刚刚写回了什么 | `updates` |
| 每一步之后完整 State 是什么 | `values` |
| 节点主动推送的进度或审计事件 | `custom` |
| 模型逐 token 输出 | `messages` |

`version="v2"` 的价值是统一事件结构：

```python
{
    "type": "updates" | "values" | "custom" | "messages",
    "ns": (...),
    "data": ...,
}
```

在安全巡检场景里：

- `custom` 适合展示“检查到哪一步”；
- `updates` 适合确认“哪个节点写入了哪些发现”；
- `messages` 适合观察“本地 LLM 正在如何生成风险判断”；
- `values` 适合回看“完整巡检状态是否逐步成形”。

后面学习循环、Send 并行、子图和多 Agent 时，我们都可以用同一套 streaming 观察方法来确认图到底怎么跑。